In [ ]:
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer

import webbrowser

In [ ]:
file_full = "../data/data.parquet"
file_min = "../data/min_data_format.parquet"

In [ ]:
df_full = pd.read_parquet(file_full)
df_min = pd.read_parquet(file_min)

check differences between data (df_full) and min_data_format (df_min) and decide with which to continue

In [ ]:
list(df_full.columns)

In [ ]:
list(df_min.columns)

In [ ]:
[c for c in list(df_full.columns) if c not in list(df_min.columns)]

df_min seems sufficient for analysis

In [ ]:
df_min

# Questions for EDA

## Which programms aim at funding democracy?

In [ ]:
df_min_demo = df_min.loc[df_min["description"].str.lower().str.contains("demokrati")]
df_min_demo

In [ ]:
# # Open fund websites for exemplary funds
# for url in df_min_demo['url'].head(10):
#     webbrowser.open(url)

## Which locations are how often funded?

In [ ]:
funding_locations = df_min["funding_location"].value_counts()
funding_locations

note: there are very few programs which fund multiple locations

-> wide format may be necessary

## Which funding types are given how often?

In [ ]:
funding_types = df_min["funding_type"].value_counts()
funding_types

note: there are very few programms which support different types

-> wide format may be necessary

## Create exemplary wide formats

In [ ]:
bundesland_abbr = {
    "Bayern": "BY",
    "Baden-Württemberg": "BW",
    "Berlin": "BE",
    "Brandenburg": "BB",
    "Bremen": "HB",
    "Hamburg": "HH",
    "Hessen": "HE",
    "Mecklenburg-Vorpommern": "MV",
    "Niedersachsen": "NI",
    "Nordrhein-Westfalen": "NW",
    "Rheinland-Pfalz": "RP",
    "Saarland": "SL",
    "Sachsen": "SN",
    "Sachsen-Anhalt": "ST",
    "Schleswig-Holstein": "SH",
    "Thüringen": "TH"
}

In [ ]:
def binarize(df: pd.DataFrame, column: str):
    """
    Convert comma-separated funding_location values to binary indicator columns.
    Where 'bundesweit' is 1, fill all location columns (except 'Sonstige') with 1.

    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame containing 'funding_location' column
        
    Returns:
    --------
    pd.DataFrame
        Original dataframe with added binary location columns
    """
    mlb = MultiLabelBinarizer()
    column_split = df[column].fillna('').str.split(',').apply(lambda x:
    [s.strip() for s in x if s.strip()])
    df_wide = pd.DataFrame(mlb.fit_transform(column_split),
                                    columns=mlb.classes_,
                                    index=df.index)
    # Reindex to include all original rows (those without locations will have 0s)
    df_wide = df_wide.reindex(df.index, fill_value=0)

    # Convert to int to get true binary values
    df_wide = df_wide.astype(int)

    if column == 'funding_location':
        # Fill all location columns (except "Sonstige") with 1 where "bundesweit" is 1
        if 'bundesweit' in df_wide.columns:
            # Get all location columns except "Sonstige"
            location_cols = [col for col in df_wide.columns if col not in ['bundesweit', 'Sonstige']]
            # Set all these columns to 1 where bundesweit is 1
            df_wide.loc[df_wide['bundesweit'] == 1, location_cols] = 1
            df_wide = df_wide.drop(columns='bundesweit')

        df_wide = df_wide.rename(mapper=bundesland_abbr, axis=1)

    df_wide = df_wide[df_wide.columns].add_prefix(prefix= column + "_", axis=1)

    # Add back to original dataframe
    df_result = pd.concat([df, df_wide], axis=1)
    df_result = df_result.drop(columns=column)

    return df_result

In [ ]:
df_binarized = binarize(df_min, 'funding_location')
# df_binarized = binarize(df_binarized, 'funding_type')
df_binarized

In [ ]:
df_binarized = binarize(df_binarized, 'funding_type')
df_binarized

TODO NEXT: see notes -> create minimalistic dashboard with map representing locations, funding type as filter and table for overview of all fundings